#### 1. Librerías.

In [65]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [66]:
%run "./constantes/constantes.ipynb"

#### 3. Funciones.

In [67]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [68]:
#a. Train.
df_train = pd.read_csv(path_train_fe)

In [69]:
#b. Test.
df_test = pd.read_csv(path_test_crudo)

In [70]:
#c. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir)

In [71]:
#d. Libros y Lectores (lo tomo para armar df_test).
df_libros = pd.read_csv(path_libros_fe)
df_lectores = pd.read_csv(path_lectores_fe)

#### 5. Preparación previa.

In [72]:
#a. Forma final.
#i. Train.
print(f"Train: {df_train.shape}")
print("\n")
#ii. Test.
print(f"Test: {df_test.shape}")

Train: (383547, 122)


Test: (77860, 4)


In [ ]:
#b. Me aseguro que no hayan quedado nulos en los ratings.
cols_rating = [c for c in df_train.columns if c.startswith("rating_prom")]
print("Train.")
print(df_train[cols_rating].isna().sum())

Train.
rating_prom_id_lector                          0
rating_prom_id_lector_autor                    0
rating_prom_id_lector_genero_libro_agrupado    0
rating_prom_id_libro                           0
rating_prom_autor                              0
rating_prom_genero                             0
dtype: int64


Test.
Series([], dtype: float64)


In [74]:
#c. Armo X e y. 
features_base = [
    "anio_edicion", 
    "nacimiento",
    "edad_al_interactuar", 
    #"dias_transcurridos_interaccion",
    "anios_transcurridos_edicion", 
    #"antiguedad_libro_hoy",
    'frecuencia_lector', 
    'frecuencia_libro', 
    'n_interacciones_lector_autor',
    'n_interacciones_lector_genero', 
    'n_lectores_distintos_autor',
    #'rating_prom_id_lector', 
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado', 
    #'rating_prom_id_libro',
    #'rating_prom_autor', 
    #'rating_prom_genero', 
    'n_autores_distintos_lector',
    'n_generos_distintos_lector'
]
features_dummies = [c for c in df_train.columns if c.startswith((
    "genero_persona_", 
    #"genero_libro_agrupado_", 
    #"editorial_agrupada_", 
    #"pais_persona_agrupado"
))]
features = features_base + features_dummies

X = df_train[features]
y = df_train["rating"]

In [75]:
#d. Split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### 6. Entrenamiento.

In [76]:
#a. Busco hiperparametros con RandomizedSearchCV.
#i. Espacio de búsqueda.
#param_dist = {
#    "n_estimators": [100, 200, 500],
#    "max_depth": [10, 20, 30],
#    "min_samples_leaf": [3, 5, 10],
#    "min_samples_split": [5, 10],
#    "max_features": ["sqrt", "log2"],
#}

#ii. Armo la búsqueda.
#busqueda = RandomizedSearchCV(
#    estimator=RandomForestRegressor(random_state=42, n_jobs=1),
#    param_distributions=param_dist,
#    n_iter=30,
#    scoring="neg_root_mean_squared_error",
#    cv=3,
#    random_state=42,
#    verbose=2,
#    n_jobs=-1,
#)

#iii. Corro la búsqueda sobre train.
#busqueda.fit(X_train, y_train)

#iv. Reviso los mejores hiperparámetros encontrados.
#print("Mejores hiperparámetros:", busqueda.best_params_)
#print("Mejor RMSE (CV, promedio de los 3 folds):", -busqueda.best_score_)

# Si quiero dejar congelado los hiperparámetros.
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_leaf=3,
    min_samples_split=10,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

,n_estimators,500
,criterion,'squared_error'
,max_depth,30
,min_samples_split,10
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


#### 7. Evaluación del modelo sobre test - RMSE.

In [ ]:
#a. Predigo test.
#rf = busqueda.best_estimator_ # Descomentar si hice búsqueda. Comentar si están congelados los hiperparámetros.
#preds = rf.predict(X_test)

In [ ]:
#b. Evaluo el RMSE Global.
#i. Realizao predicciones sobre df_test.
#df_test['pred'] = rf.predict(df_test[X.columns])

#ii. Calculo la métrica RMSE.
#rmse_test = np.sqrt(mean_squared_error(df_test[y.name], df_test['pred']))
#print(f"RMSE en Test: {rmse_test:.4f}")

#### 8. Evaluación del modelo sobre test -  nDCG@20.

In [81]:
#a. Precómputos (fuera del loop).
#1. Universo de libros.
conn = sqlite3.connect(path_db)
todos_los_libros = pd.read_sql("SELECT id_libro FROM interacciones", conn)["id_libro"].unique()
conn.close()
#2. Historial por lector.
leidos_por_lector = df_train[["id_lector","id_libro"]].groupby("id_lector")["id_libro"].apply(set).to_dict()
#3. Ground truth por lector.
gt_por_lector = {lid: pd.Series(g["rating"].values, index=g["id_libro"].values)
                 for lid, g in df_test.groupby("id_lector")}
#4. Lectores de test a recomendarle.
id_lectores_test = df_test["id_lector"].unique()

In [82]:
#b. Variables que me van a servir para el feature engineering de test.
#i.Media global del rating en TRAIN.
media_global = df_train["rating"].mean()

#ii. Características del lector.
x_caract_lector_base = [
    "id_lector",
    #"nombre",
    #"vive_en",
    "nacimiento",
    "ciudad",
    "pais"
] 

caract_lector_base = df_lectores[x_caract_lector_base].drop_duplicates("id_lector")

# Las dummies las armo desde df_lectores (tiene TODOS los lectores), no desde df_train
# (solo tiene los que interactuaron). Si no, todo lector sin historial queda en NaN.
caract_lector_dummies = pd.get_dummies(
    df_lectores[["id_lector", "genero_persona"]].drop_duplicates("id_lector"),
    columns=["genero_persona"]
)

# Alineo con las columnas exactas que vio el modelo en train:
# si aparece una categoría que train no tenía, la descarto; si falta una que espera, la creo en 0.
cols_dummies_train = [c for c in df_train.columns if c.startswith("genero_persona_")]
caract_lector_dummies = caract_lector_dummies.reindex(
    columns=["id_lector"] + cols_dummies_train,
    fill_value=0
)

caract_lector = caract_lector_base.merge(caract_lector_dummies,how="left",on="id_lector")

# Red de seguridad: si algún lector no matcheó en el merge, las dummies quedan en 0.
caract_lector[cols_dummies_train] = caract_lector[cols_dummies_train].fillna(0)

# Frecuencia del lector para TEST ----> Se calcula sobre todo df_train, sin LOO.
freq_lector_test = (
    df_train
    .groupby("id_lector")
    .size()
)

caract_lector["frecuencia_lector"] = (
    caract_lector["id_lector"]
    .map(freq_lector_test)
    .fillna(0)
)

# Rating promedio del lector para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_lector_test = (
    df_train
    .groupby("id_lector")["rating"]
    .mean()
)

caract_lector["rating_prom_id_lector"] = (
    caract_lector["id_lector"]
    .map(means_lector_test)
    .fillna(media_global)
)

# Cantidad de autores distintos que leyó para TEST ----> Se calcula sobre todo df_train, sin LOO.
autores_por_lector_test = (
    df_train
    .groupby("id_lector")["autor"]
    .nunique()
)

caract_lector["n_autores_distintos_lector"] = (
    caract_lector["id_lector"]
    .map(autores_por_lector_test)
    .fillna(0)
)

# Cantidad de géneros distintos que leyó para TEST ----> Se calcula sobre todo df_train, sin LOO.
generos_por_lector_test = (
    df_train
    .groupby("id_lector")["genero_libro_agrupado"]
    .nunique()
)

caract_lector["n_generos_distintos_lector"] = (
    caract_lector["id_lector"]
    .map(generos_por_lector_test)
    .fillna(0)
)

#iii. Comprobación: no puede quedar ningún nulo.
nulos = caract_lector.drop(columns=["ciudad", "pais"]).isna().sum()
print("Nulos en caract_lector:")
print(nulos[nulos > 0] if nulos.sum() else "Ninguno.")
print("Lectores:", len(caract_lector), "| Duplicados:", caract_lector["id_lector"].duplicated().sum())

Nulos en caract_lector:
Ninguno.
Lectores: 11285 | Duplicados: 0


In [83]:
#iii. Características de los libros.
x_caract_libros_base = [
    "id_libro",
    "autor",
    "genero_libro_agrupado",
    #"editorial",
    "anio_edicion",
    #"isbn",
    #"resumen"
]
caract_libros_base = df_libros[x_caract_libros_base].drop_duplicates("id_libro")

# Las dummies las armo desde df_libros (tiene TODO el catálogo), no desde df_train
# (solo tiene los libros que alguien leyó). Si no, todo libro sin interacciones queda en NaN.
caract_libros_dummies = pd.get_dummies(
    df_libros[["id_libro", "genero_libro_agrupado"]].drop_duplicates("id_libro"),
    columns=["genero_libro_agrupado"]
)

# Alineo con las columnas exactas que vio el modelo en train:
# si aparece una categoría que train no tenía, la descarto; si falta una que espera, la creo en 0.
cols_dummies_libro_train = [c for c in df_train.columns if c.startswith("genero_libro_agrupado_")]
caract_libros_dummies = caract_libros_dummies.reindex(
    columns=["id_libro"] + cols_dummies_libro_train,
    fill_value=0
)

caract_libros = caract_libros_base.merge(caract_libros_dummies,how="left",on="id_libro")

# Red de seguridad: si algún libro no matcheó en el merge, las dummies quedan en 0.
caract_libros[cols_dummies_libro_train] = caract_libros[cols_dummies_libro_train].fillna(0)

# Frecuencia del libro para TEST ----> Se calcula sobre todo df_train, sin LOO.
freq_libro_test = (
    df_train
    .groupby("id_libro")
    .size()
)

caract_libros["frecuencia_libro"] = (
    caract_libros["id_libro"]
    .map(freq_libro_test)
    .fillna(0)
)

# Cantidad de lectores distintos del autor para TEST ----> Se calcula sobre todo df_train, sin LOO.
pop_autor_test = (
    df_train
    .groupby("autor")["id_lector"]
    .nunique()
)

caract_libros["n_lectores_distintos_autor"] = (
    caract_libros["autor"]
    .map(pop_autor_test)
    .fillna(0)
)

# Rating promedio del libro para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_libro_test = (
    df_train
    .groupby("id_libro")["rating"]
    .mean()
)

caract_libros["rating_prom_id_libro"] = (
    caract_libros["id_libro"]
    .map(means_libro_test)
    .fillna(media_global)
)

# Rating promedio del autor para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_autor_test = (
    df_train
    .groupby("autor")["rating"]
    .mean()
)

caract_libros["rating_prom_autor"] = (
    caract_libros["autor"]
    .map(means_autor_test)
    .fillna(media_global)
)

# Rating promedio del género para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_genero_test = (
    df_train
    .groupby("genero_libro_agrupado")["rating"]
    .mean()
)

caract_libros["rating_prom_genero"] = (
    caract_libros["genero_libro_agrupado"]
    .map(means_genero_test)
    .fillna(media_global)
)

#iv. Comprobación: no puede quedar ningún nulo en lo que va al modelo.
nulos = caract_libros.isna().sum()
print("Nulos en caract_libros:")
print(nulos[nulos > 0] if nulos.sum() else "Ninguno.")
print("Libros:", len(caract_libros), "| Duplicados:", caract_libros["id_libro"].duplicated().sum())
print("Libros sin interacciones en train:", (caract_libros["frecuencia_libro"] == 0).sum())

Nulos en caract_libros:
autor                    78227
genero_libro_agrupado    78220
dtype: int64
Libros: 128743 | Duplicados: 0
Libros sin interacciones en train: 84772


In [84]:
#vi. Afinidad lector-autor para TEST ----> Se calcula sobre todo df_train, sin LOO.
afinidad_lector_autor_test = (
    df_train
    .groupby(["id_lector", "autor"])
    .agg(
        rating_prom_id_lector_autor=("rating", "mean"),
        n_interacciones_lector_autor=("rating", "count")
    )
    .reset_index()
)

#v. Afinidad lector-género para TEST ----> Se calcula sobre todo df_train, sin LOO.
afinidad_lector_genero_test = (
    df_train
    .groupby(["id_lector", "genero_libro_agrupado"])
    .agg(
        rating_prom_id_lector_genero_libro_agrupado=("rating", "mean"),
        n_interacciones_lector_genero=("rating", "count")
    )
    .reset_index()
)

In [85]:
#vii. Comprobación.
print(
    "Duplicados lectores:",
    caract_lector["id_lector"].duplicated().sum()
)

print(
    "Duplicados libros:",
    caract_libros["id_libro"].duplicated().sum()
)

print(
    "Duplicados lector-autor:",
    afinidad_lector_autor_test
    .duplicated(["id_lector", "autor"])
    .sum()
)

print(
    "Duplicados lector-género:",
    afinidad_lector_genero_test
    .duplicated(["id_lector", "genero_libro_agrupado"])
    .sum()
)

Duplicados lectores: 0
Duplicados libros: 0
Duplicados lector-autor: 0
Duplicados lector-género: 0


In [86]:
lectores_test_muestra = (
    df_test["id_lector"]
    .drop_duplicates()
    .sample(
        n=1000,
        random_state=42
    )
    .tolist()
)

In [ ]:
#c. Calculamos.
#i. Lista vacía donde iré almacenando los nDCG de cada usuario, y datos para luego monitorear.
ndcg_lista = []
total_lectores = len(lectores_test_muestra)
print("Comienza la predicción general.")
#ii. Recorro cada id_lector a recomendar.
for i, id_lector in enumerate(lectores_test_muestra, start=1):
    print("\nLector: {} {}/{}".format(id_lector, i, total_lectores))
    #1. Me traigo los libros a recomendarle al id_lector.
    #print("1. Retrieval.")
    libros_candidatos_a_recomendar = retrieval(id_lector)

    #2. Me traigo el Ground Truth del lector (que por diseño, son 20 en test).
    #print("2. Me traigo el True Relevance.")
    true_relevance = gt_por_lector[id_lector]

    #3. Realizo el feature engineering sobre todos los libros candidatos a recomendar.
    #print("3. Armo las features de los libros candidatos.")
    df_features_candidatos = feature_engineering_test(id_lector, libros_candidatos_a_recomendar)

    #4. Predigo el ranking para cada libro candidato del id_lector
    #print("4. Predigo sobre los libros candidatos su rating.")
    predicted_scores_dict = ranking(df_features_candidatos, features, rf)

    #5. Creo una lista de los libros verdaderos + los que evalué, universo en común a evaluar.
    id_libros = list(set(true_relevance.index) |set(predicted_scores_dict.keys()))

    #6. Traigo el rating real de cada libro. Si no es relevante, entonces = 0.
    y_true = np.asarray([[true_relevance.get(id_libro, 0) for id_libro in id_libros]])

    #7. Traigo el rating predicho por el modelo de cada libro. Si no lo evaluó, entonces = 0.
    y_score = np.asarray([[predicted_scores_dict.get(id_libro, 0) for id_libro in id_libros]])

    #8. Calculo el nDCG@20 para el id_lector.
    #print("5. Calculo el nDCG para el lector en cuestión.")
    ndcg = ndcg_score(y_true, y_score, k=20)

    #9. Lo agrego a la lista.
    ndcg_lista.append(ndcg)

    #10. Imprimo el nDCG del id_lector.
    print("El nCDG es de:{}".format(ndcg))
    
#iii. Imprimo el nDCG promedio de todos los id_lectores a predecir.
ndcg_arr = np.array(ndcg_lista)
print("\nEl nDCG@20 promedio es: {:.4f} ± {:.4f} (SE)".format(ndcg_arr.mean(), ndcg_arr.std(ddof=1) / np.sqrt(len(ndcg_arr))))

Comienza la predicción general.

Lector: susy-berciano 1/1000
El nCDG es de:0.0

Lector: mariamag 2/1000
El nCDG es de:0.04083705652688734

Lector: zaina 3/1000
El nCDG es de:0.0542668252584633

Lector: epeich 4/1000
El nCDG es de:0.0

Lector: anadiomena 5/1000
El nCDG es de:0.12172497467229267

Lector: helenadalmau 6/1000
El nCDG es de:0.0785703393740845

Lector: andrixo 7/1000
El nCDG es de:0.0

Lector: aitana_munoz 8/1000
El nCDG es de:0.09029091316047366

Lector: mageni 9/1000
El nCDG es de:0.03255160027081613

Lector: artdyl 10/1000
El nCDG es de:0.0

Lector: katagata 11/1000
El nCDG es de:0.0545348132961354

Lector: henri-mesquida 12/1000
El nCDG es de:0.0

Lector: jamoreno 13/1000
El nCDG es de:0.0

Lector: bgo 14/1000
El nCDG es de:0.046049145154732234

Lector: alberto-acevedo 15/1000
El nCDG es de:0.0

Lector: 10155998755965659 16/1000
El nCDG es de:0.229158937310006

Lector: el-escondite 17/1000
El nCDG es de:0.17890929235673553

Lector: elena 18/1000
El nCDG es de:0.04605403

#### 9. Exportación del modelo.

In [ ]:
#a. Exportamos el modelo entrenado.
joblib.dump(rf, path_modelo)
print("Modelo exportado con éxito.")

Modelo exportado con éxito.


#### 10. Análisis.

In [90]:
#a. Diagnóstico: dónde gana y dónde pierde el modelo.
freq = df_train.groupby("id_lector").size()

res = pd.DataFrame({"id_lector": lectores_test_muestra, "ndcg": ndcg_lista})
res["freq"] = res["id_lector"].map(freq).fillna(0)
res["bucket"] = pd.cut(res["freq"], [-1, 0, 20, 50, 100, 250, np.inf],
                       labels=["cold", "1-20", "21-50", "51-100", "101-250", "250+"])

print(res.groupby("bucket", observed=True)["ndcg"].agg(["mean", "median", "count"]))
print("\nLectores con nDCG = 0:", (res["ndcg"] == 0).mean())

             mean    median  count
bucket                            
cold     0.000622  0.000000     25
1-20     0.089845  0.037608    278
21-50    0.078954  0.030866    246
51-100   0.065149  0.000000    198
101-250  0.030650  0.000000    188
250+     0.037019  0.000000     65

Lectores con nDCG = 0: 0.558


In [91]:
#b. Techo de recall: ¿los 20 libros del GT son siquiera candidatos?
recalls = []
for lid in lectores_test_muestra[:200]:
    cands = set(retrieval(lid))
    gt = set(gt_por_lector[lid].index)
    recalls.append(len(cands & gt) / len(gt))
print("Recall del retrieval:", np.mean(recalls))

Recall del retrieval: 1.0


In [ ]:
# Comentario.
#Para un lector fijo, el score de un candidato lo deciden rating_prom_id_lector_autor, 
# n_interacciones_lector_autor y las dos de género. 
# Un lector con 15 libros leyó quizá 12 autores: el modelo tiene que elegir entre "autor que 
# ya leyó" (12 candidatos, señal fuerte) y "autor desconocido" 
# (todo el resto, n=0 y rating=media_global). 
# Es una decisión casi binaria y acierta seguido, porque la gente vuelve a sus autores.

#Un lector con 200 libros leyó 150 autores. Ahora hay 150 candidatos con señal positiva y el modelo
#  tiene que ordenarlos entre sí. 
# Eso ya no lo resuelve n_interacciones_lector_autor, que satura. 
# Y las features que podrían desempatar —calidad del libro, popularidad, novedad— son las tres que
#  tenés comentadas.

#O sea: tus features distinguen bien "conocido vs desconocido" y no distinguen nada dentro de 
# "conocido". El lector pesado vive enteramente dentro de esa segunda categoría.